## Data Cleaning

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
billings = pd.read_csv('../../data/raw/billings.csv', low_memory=False)

In [ ]:
DROP_COLS = [
    # 100% null
    'Last_Years_Date_Paid',
    # 93% null
    'Connection_Net', 'Connection_Qty',
    'Starting_Connection_Net', 'Starting_Connection_Qty',
    # 89% null — will engineer binary discount flag
    'Discount_Amount',
    # Redundant price sub-components
    'Starting_Net', 'Starting_Vat', 'Starting_Gross',
    'Starting_Membership_Net', 'Starting_Package_Net', 'Starting_PQQ_Net',
    'Gross', 'Membership_Net', 'Package_Net', 'PQQNet',
    'Total_Amount',
    # Duplicate of Connection_Group
    'Anchor_Group',
    # Admin / redundant date cols
    'Proforma_Date',
    'DateTime_Out',
]

# Only drop columns that actually exist
DROP_COLS = [c for c in DROP_COLS if c in billings.columns]

# Also drop Unnamed cols if present
unnamed_cols = [c for c in billings.columns if 'Unnamed' in str(c)]
DROP_COLS += unnamed_cols

billings.drop(columns=DROP_COLS, inplace=True)

print(f'Dropped {len(DROP_COLS)} columns')
print(f'Shape after drop: {billings.shape}')
print(f'\nRemaining columns ({len(billings.columns)}):')
print(billings.columns.tolist())

Dropped 20 columns
Shape after drop: (122082, 39)

Remaining columns (39):
['Co_Ref', 'Renewal_Month', 'Sustainability_Score', 'Total_Renewal_Score_New', 'Last_Years_Price', 'Auto_Renewal_Score', 'Status_Scores', 'Anchoring_Score', 'Tenure_Scores', 'Proforma_Auto_Renewal', 'Proforma_World_Pay_Token', 'Current_Anchorings', 'Current_Anchor_List', 'Payment_Timeframe', 'Registration_Date', 'Proforma_Account_Stage', 'Proforma_Audit_Status', 'Current_Auto_Renewal_Flag', 'Current_World_Pay_Token', 'Renewal_Score_At_Release', 'Proforma_Membership_Status', 'Proforma_Approved_Lists', 'Tenure_Years', 'Band', 'Prospect_Renewal_Date', 'Closed_Date', 'Prospect_Status', 'Total_Net_Paid', 'Prospect_Outcome', 'Payment_Method', 'Amount', 'Connection_Group', 'Tenure_Group', '#_of_Connection', 'Last_Renewal', 'Last_Band', 'Last_Total_Net_Paid', 'Last_Connections', 'Renewal_Year']


### Remove Open records
Open prospects have no confirmed outcome yet. We cannot label them as Won or Churned,
so they **cannot be used for model training**. We drop them here.

In [ ]:
print('Prospect_Outcome before filter:')
print(billings['Prospect_Outcome'].value_counts())

billings = billings[billings['Prospect_Outcome'].isin(['Won', 'Churned'])].copy()

print(f'\nShape after removing Open: {billings.shape}')
print('Prospect_Outcome after filter:')
print(billings['Prospect_Outcome'].value_counts())

Prospect_Outcome before filter:
Prospect_Outcome
Won        101226
Churned     12668
Open         8188
Name: count, dtype: int64

Shape after removing Open: (113894, 39)
Prospect_Outcome after filter:
Prospect_Outcome
Won        101226
Churned     12668
Name: count, dtype: int64


### Remove Bad Renewal_Year outliers
Years 2027 (2 rows) and 2050 (3 rows) are clear data entry errors.

In [ ]:
print('Renewal_Year before filter:')
print(billings['Renewal_Year'].value_counts().sort_index())

bad_years = billings[~billings['Renewal_Year'].isin([2023, 2024, 2025, 2026])]
print(f'\nRows with bad Renewal_Year: {len(bad_years)}')
print(bad_years[['Co_Ref','Renewal_Year','Amount','Prospect_Outcome']])

billings = billings[billings['Renewal_Year'].isin([2023, 2024, 2025, 2026])].copy()
print(f'\nShape after removing bad years: {billings.shape}')

Renewal_Year before filter:
Renewal_Year
2023    33925
2024    34601
2025    36463
2026     8901
2027        2
2050        2
Name: count, dtype: int64

Rows with bad Renewal_Year: 4
        Co_Ref  Renewal_Year  Amount Prospect_Outcome
59026   YG4840          2027       0              Won
78362   PU4860          2027       0          Churned
114216  UQ7012          2050       0              Won
114249  RE5625          2050       0              Won

Shape after removing bad years: (113890, 39)


### Remove Zero-Amount Duplicate Entries

In [ ]:
print('Zero-amount rows:')
zero_amt = billings[billings['Amount'] == 0]
print(f'Total: {len(zero_amt)}')
print(zero_amt['Prospect_Status'].value_counts())

# Only drop zero-amount rows that are flagged as Duplicate Entry
# Keep zero-amount Won/Churned rows — they may be legitimate edge cases
mask_dup = (billings['Amount'] == 0) & (billings['Prospect_Status'] == 'Duplicate Entry')
print(f'\nDuplicate Entry zero-amount rows to drop: {mask_dup.sum()}')

billings = billings[~mask_dup].copy()
print(f'Shape after removing zero-amount duplicates: {billings.shape}')

Zero-amount rows:
Total: 121
Prospect_Status
Duplicate Entry                             59
Renewed                                     36
Application and Money In                    14
Do Not Work for Client                       3
Not Affordable                               1
Not Value for Money                          1
Existing Safecontractor Member               1
No Longer Trading                            1
Paid                                         1
Insufficient Contract Value                  1
Promised to Pay - Fees not Received          1
Decline 3rd party accreditation requests     1
Poor Customer Service                        1
Name: count, dtype: int64

Duplicate Entry zero-amount rows to drop: 59
Shape after removing zero-amount duplicates: (113831, 39)


In [ ]:
DATE_COLS = [
    'Prospect_Renewal_Date',
    'Registration_Date',
    'Closed_Date',
    'Last_Renewal',
    'Renewal_Month',
]
DATE_COLS = [c for c in DATE_COLS if c in billings.columns]

for col in DATE_COLS:
    billings[col] = pd.to_datetime(billings[col], format='%d-%m-%Y', errors='coerce')

print('Date columns parsed. Null counts after parsing:')
print(billings[DATE_COLS].isnull().sum())
print()
print('Sample:')
print(billings[DATE_COLS].head(3))

Date columns parsed. Null counts after parsing:
Prospect_Renewal_Date        0
Registration_Date          974
Closed_Date                  0
Last_Renewal             46650
Renewal_Month                0
dtype: int64

Sample:
  Prospect_Renewal_Date Registration_Date Closed_Date Last_Renewal  \
0            2024-11-05        2021-11-05  2024-11-05   2023-11-01   
1            2025-08-09        2024-08-09  2025-08-09          NaT   
2            2025-03-12        2015-03-12  2025-03-12   2024-03-01   

  Renewal_Month  
0    2024-11-01  
1    2025-08-01  
2    2025-03-01  


### Standardise Flag Columns (y/n → 0/1)

In [ ]:
# Current_Auto_Renewal_Flag: y=1, n=0
print('Current_Auto_Renewal_Flag raw:', billings['Current_Auto_Renewal_Flag'].value_counts().to_dict())
billings['auto_renewal'] = (billings['Current_Auto_Renewal_Flag'] == 'y').astype(int)
print('auto_renewal:', billings['auto_renewal'].value_counts().to_dict())

# Current_World_Pay_Token: y=1, n=0
print('\nCurrent_World_Pay_Token raw:', billings['Current_World_Pay_Token'].value_counts().to_dict())
billings['has_worldpay_token'] = (billings['Current_World_Pay_Token'] == 'y').astype(int)
print('has_worldpay_token:', billings['has_worldpay_token'].value_counts().to_dict())

Current_Auto_Renewal_Flag raw: {'y': 106799, 'n': 7032}
auto_renewal: {1: 106799, 0: 7032}

Current_World_Pay_Token raw: {'n': 60112, 'y': 53719}
has_worldpay_token: {0: 60112, 1: 53719}


### Last_Band / Last_Years_Price / Last_Connections
These are null for **new customers** (first-ever renewal — no prior year exists).
This is **structural** missingness, not a data error.

In [ ]:
print('Last_Band nulls:', billings['Last_Band'].isnull().sum())
print('Last_Years_Price nulls:', billings['Last_Years_Price'].isnull().sum())
print('Last_Connections nulls:', billings['Last_Connections'].isnull().sum())
print()
print('These nulls are expected for new customers (Tenure_Years = 1).')
print('Tenure_Years == 1 count:', (billings['Tenure_Years'] == 1).sum())

# Fill Last_Years_Price with 0 for new customers (no prior price)
billings['Last_Years_Price'] = billings['Last_Years_Price'].fillna(0)

# Fill Last_Connections with 0 for new customers
if 'Last_Connections' in billings.columns:
    billings['Last_Connections'] = billings['Last_Connections'].fillna(0)

# Last_Band — fill with 'New Customer' category
billings['Last_Band'] = billings['Last_Band'].fillna('New Customer')

print('\nAfter imputation:')
print('Last_Years_Price nulls:', billings['Last_Years_Price'].isnull().sum())
print('Last_Band nulls:', billings['Last_Band'].isnull().sum())

Last_Band nulls: 46667
Last_Years_Price nulls: 8809
Last_Connections nulls: 46724

These nulls are expected for new customers (Tenure_Years = 1).
Tenure_Years == 1 count: 17842

After imputation:
Last_Years_Price nulls: 0
Last_Band nulls: 0


### Tenure_Years
Small number of nulls (~1%). Fill with median tenure.

In [ ]:
print('Tenure_Years nulls before:', billings['Tenure_Years'].isnull().sum())
print('Tenure_Years stats:\n', billings['Tenure_Years'].describe().round(2))

# Cap extreme values at 30 (confirmed no valid tenure > 30 in this dataset)
billings['Tenure_Years'] = billings['Tenure_Years'].clip(upper=30)

median_tenure = billings['Tenure_Years'].median()
print(f'\nMedian tenure used for imputation: {median_tenure}')
billings['Tenure_Years'] = billings['Tenure_Years'].fillna(median_tenure)

print('Tenure_Years nulls after :', billings['Tenure_Years'].isnull().sum())

Tenure_Years nulls before: 974
Tenure_Years stats:
 count    112857.00
mean          6.86
std           5.39
min           0.00
25%           2.00
50%           5.00
75%          10.00
max          26.00
Name: Tenure_Years, dtype: float64

Median tenure used for imputation: 5.0
Tenure_Years nulls after : 0


### Categorical Columns: Fill nulls with 'Unknown'

In [ ]:
CAT_FILL_UNKNOWN = [
    'Proforma_Account_Stage',
    'Proforma_Membership_Status',
    'Proforma_Audit_Status',
    'Tenure_Group',
    'Connection_Group',
    'Band',
    'Registration_Date',  # ~1% null dates — leave as NaT, handle in feature eng
]
CAT_FILL_UNKNOWN = [c for c in CAT_FILL_UNKNOWN if c in billings.columns]

for col in CAT_FILL_UNKNOWN:
    if billings[col].dtype == 'object':
        before = billings[col].isnull().sum()
        billings[col] = billings[col].fillna('Unknown')
        print(f'{col}: {before} nulls → filled with "Unknown"')

Proforma_Account_Stage: 9162 nulls → filled with "Unknown"
Proforma_Membership_Status: 59 nulls → filled with "Unknown"
Proforma_Audit_Status: 9162 nulls → filled with "Unknown"
Tenure_Group: 974 nulls → filled with "Unknown"
Connection_Group: 59 nulls → filled with "Unknown"
Band: 6 nulls → filled with "Unknown"


### Numeric Columns: Fill with 0 or Median

In [ ]:
# #_of_Connection — fill with 0 (no connections = independent)
print('#_of_Connection nulls before:', billings['#_of_Connection'].isnull().sum())
billings['#_of_Connection'] = billings['#_of_Connection'].fillna(0)
print('#_of_Connection nulls after :', billings['#_of_Connection'].isnull().sum())


# Renewal_Score_At_Release — fill with median
if 'Renewal_Score_At_Release' in billings.columns:
    med = billings['Renewal_Score_At_Release'].median()
    print(f'\nRenewal_Score_At_Release nulls before: {billings["Renewal_Score_At_Release"].isnull().sum()}')
    billings['Renewal_Score_At_Release'] = billings['Renewal_Score_At_Release'].fillna(med)
    print(f'Renewal_Score_At_Release nulls after : {billings["Renewal_Score_At_Release"].isnull().sum()}')

# Payment_Timeframe — fill with 0 (0 = paid on time)
if 'Payment_Timeframe' in billings.columns:
    print(f'\nPayment_Timeframe nulls before: {billings["Payment_Timeframe"].isnull().sum()}')
    billings['Payment_Timeframe'] = billings['Payment_Timeframe'].fillna(0)
    print(f'Payment_Timeframe nulls after : {billings["Payment_Timeframe"].isnull().sum()}')


#_of_Connection nulls before: 59
#_of_Connection nulls after : 0

Renewal_Score_At_Release nulls before: 59
Renewal_Score_At_Release nulls after : 0

Payment_Timeframe nulls before: 12608
Payment_Timeframe nulls after : 0


In [ ]:
# Total_Net_Paid — fill with Amount (if not paid yet, use the asked amount)
if 'Total_Net_Paid' in billings.columns:
    print(f'\nTotal_Net_Paid nulls before: {billings["Total_Net_Paid"].isnull().sum()}')
    billings['Total_Net_Paid'] = billings['Total_Net_Paid'].fillna(billings['Amount'])
    print(f'Total_Net_Paid nulls after : {billings["Total_Net_Paid"].isnull().sum()}')


Total_Net_Paid nulls before: 12608
Total_Net_Paid nulls after : 0


In [ ]:
remaining_nulls = billings.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print('Remaining nulls after all imputation:')
if len(remaining_nulls) == 0:
    print('  None — all nulls handled!')
else:
    print(remaining_nulls.to_string())

Remaining nulls after all imputation:
Proforma_Auto_Renewal       17985
Proforma_World_Pay_Token    17985
Current_Anchor_List         23948
Registration_Date             974
Proforma_Approved_Lists        59
Last_Renewal                46650
Last_Total_Net_Paid         46700


In [ ]:
# ── Step 5g — Handle remaining nulls ────────────────────────────────

# 1. DROP redundant columns (already have better versions)
cols_to_drop = [
    'Proforma_Auto_Renewal',      # redundant with auto_renewal (engineered)
    'Proforma_World_Pay_Token',   # redundant with has_worldpay_token (engineered)
    'Current_Anchor_List',        # text list of client names — not a model feature
    'Last_Renewal',               # date column, not in final feature list
    'Last_Total_Net_Paid',        # 41% null, Last_Years_Price already covers prior pricing
]
cols_to_drop = [c for c in cols_to_drop if c in billings.columns]  # ← was already correct
billings.drop(columns=cols_to_drop, inplace=True)
print(f'Dropped {len(cols_to_drop)} redundant columns: {cols_to_drop}')

# 2. Proforma_Approved_Lists — fill 59 nulls with median
med_approved = billings['Proforma_Approved_Lists'].median()  # ← was already correct
billings['Proforma_Approved_Lists'] = billings['Proforma_Approved_Lists'].fillna(med_approved)
print(f'\nProforma_Approved_Lists: filled 59 nulls with median ({med_approved})')

# 3. days_as_member — fill 974 nulls with its own median
#    (Registration_Date stays NaT — it's not a model feature, only used to compute this)
if 'days_as_member' in billings.columns:
    med_days = billings['days_as_member'].median()  # ← FIX: was df['days_as_member']
    billings['days_as_member'] = billings['days_as_member'].fillna(med_days)
    print(f'days_as_member: filled nulls with median ({med_days:.0f} days)')

# 4. Final null check — should be zero (Registration_Date excluded — not going into FINAL_COLS)
remaining = billings[[c for c in billings.columns if c != 'Registration_Date']].isnull().sum()
remaining = remaining[remaining > 0]
print(f'\nRemaining nulls (excl. Registration_Date): {len(remaining)} columns')
if len(remaining) == 0:
    print('All nulls resolved')
else:
    print(remaining)

Dropped 0 redundant columns: []

Proforma_Approved_Lists: filled 59 nulls with median (1.0)

Remaining nulls (excl. Registration_Date): 0 columns
All nulls resolved


In [ ]:
remaining_nulls = billings.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print('Remaining nulls after all imputation:')
if len(remaining_nulls) == 0:
    print('  None — all nulls handled!')
else:
    print(remaining_nulls.to_string())

Remaining nulls after all imputation:
Registration_Date    974


In [ ]:
# Fill Registration_Date nulls by back-calculating from Prospect_Renewal_Date - Tenure_Years
# Logic: if we know renewal date and tenure, we can approximate when they registered

mask = billings['Registration_Date'].isnull()
print(f'Registration_Date nulls before: {mask.sum()}')

billings.loc[mask, 'Registration_Date'] = (
    billings.loc[mask, 'Prospect_Renewal_Date'] - 
    pd.to_timedelta(billings.loc[mask, 'Tenure_Years'] * 365.25, unit='D')
)

print(f'Registration_Date nulls after : {billings["Registration_Date"].isnull().sum()}')

# Now recompute days_as_member for those rows so it's consistent
billings['days_as_member'] = (
    billings['Prospect_Renewal_Date'] - billings['Registration_Date']
).dt.days

# Fill any still-remaining nulls with median (safety net)
billings['days_as_member'] = billings['days_as_member'].fillna(billings['days_as_member'].median())

print(f'days_as_member nulls after    : {billings["days_as_member"].isnull().sum()}')
print(f'\nRegistration_Date sample (filled rows):')
print(billings.loc[mask, ['Co_Ref','Prospect_Renewal_Date','Tenure_Years','Registration_Date']].head(5))

Registration_Date nulls before: 974
Registration_Date nulls after : 0
days_as_member nulls after    : 0

Registration_Date sample (filled rows):
     Co_Ref Prospect_Renewal_Date  Tenure_Years   Registration_Date
81   CP1006            2025-07-12           5.0 2020-07-11 18:00:00
137  CP1325            2025-07-28           5.0 2020-07-27 18:00:00
220  RV8459            2025-10-06           5.0 2020-10-05 18:00:00
335  PB7918            2024-12-01           5.0 2019-12-01 18:00:00
429  IK3789            2025-10-26           5.0 2020-10-25 18:00:00


In [ ]:
remaining_nulls = billings.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print('Remaining nulls after all imputation:')
if len(remaining_nulls) == 0:
    print('  None — all nulls handled!')
else:
    print(remaining_nulls.to_string())

Remaining nulls after all imputation:
  None — all nulls handled!


In [ ]:
billings.shape

(113831, 37)

In [ ]:
billings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 113831 entries, 0 to 122081
Data columns (total 37 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   Co_Ref                      113831 non-null  object        
 1   Renewal_Month               113831 non-null  datetime64[ns]
 2   Sustainability_Score        113831 non-null  float64       
 3   Total_Renewal_Score_New     113831 non-null  float64       
 4   Last_Years_Price            113831 non-null  float64       
 5   Auto_Renewal_Score          113831 non-null  int64         
 6   Status_Scores               113831 non-null  int64         
 7   Anchoring_Score             113831 non-null  float64       
 8   Tenure_Scores               113831 non-null  float64       
 9   Current_Anchorings          113831 non-null  int64         
 10  Payment_Timeframe           113831 non-null  float64       
 11  Registration_Date           113831 non-null 

In [ ]:
billings.columns

Index(['Co_Ref', 'Renewal_Month', 'Sustainability_Score',
       'Total_Renewal_Score_New', 'Last_Years_Price', 'Auto_Renewal_Score',
       'Status_Scores', 'Anchoring_Score', 'Tenure_Scores',
       'Current_Anchorings', 'Payment_Timeframe', 'Registration_Date',
       'Proforma_Account_Stage', 'Proforma_Audit_Status',
       'Current_Auto_Renewal_Flag', 'Current_World_Pay_Token',
       'Renewal_Score_At_Release', 'Proforma_Membership_Status',
       'Proforma_Approved_Lists', 'Tenure_Years', 'Band',
       'Prospect_Renewal_Date', 'Closed_Date', 'Prospect_Status',
       'Total_Net_Paid', 'Prospect_Outcome', 'Payment_Method', 'Amount',
       'Connection_Group', 'Tenure_Group', '#_of_Connection', 'Last_Band',
       'Last_Connections', 'Renewal_Year', 'auto_renewal',
       'has_worldpay_token', 'days_as_member'],
      dtype='object')